# Kinthread data prep on Colab (keeps the laptop free)

Run on **aravallianirudh2005** (owner of the shared `kinthread` folder). Every step is resumable and writes to Drive:
1. dataset_v7 (verified questions with the new forms) + audit, and 2. the kinship text for pre-v2: CPU work, about 1.5 h. Start on a **CPU runtime** (Run all stops at step 3).
3. STaR round 2 and re-read-and-correct rows: switch to a **T4** and Run all again (finished steps are skipped; about 30 min).

Then run `pretrain_colab.ipynb` (pre-v2) and later `train_reader_colab.ipynb` (r5).

In [ ]:
import os, torch
REPO = '/content/Kinthread-llm-model'
if not os.path.exists(REPO):
    !git clone --depth 1 -b main https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
from google.colab import drive
drive.mount('/content/drive')
assert os.path.ismount('/content/drive'), 'Google Drive did not mount: click Allow and run this cell again'
BASE = next(b for b in ('/content/drive/MyDrive/kinthread_shared', '/content/drive/MyDrive/kinthread') if os.path.isdir(b) or b.endswith('/kinthread'))
D = f'{BASE}/data'
os.makedirs(D, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none', '| data ->', D)

In [ ]:
# 1. dataset_v7 (skipped when done): verified pool with the locked deep test of v5, then audited and cleaned
if not os.path.exists(f'{D}/dataset_v7/manifest.json'):
    if not os.path.exists(f'{D}/dataset_v7_raw/manifest.json'):
        !python build_dataset.py --root {D}/dataset_v7_raw --workers 2
    !rm -rf {D}/dataset_v7_raw/test && cp -r data/dataset_v5/test {D}/dataset_v7_raw/test
    !python audit_dataset.py --root {D}/dataset_v7_raw --clean --out {D}/dataset_v7 --block 448
!python build_dataset.py --verify --root {D}/dataset_v7

In [ ]:
# 2. Kinship text for pretraining pre-v2 (60M tokens from dataset_v7; CPU)
K = f'{BASE}/pretrain_kin_v7'
if not os.path.exists(f'{K}/manifest.json'):
    !python pretrain_data.py build --vocab data/subword_vocab.json --out {K} --tokens 60000000 --mix domain=1 --domain_src {D}/dataset_v7
!cat {K}/manifest.json | head -c 400

In [ ]:
# 3. GPU: STaR round 2 on the LLM-written questions (crawled questions stay private, on the laptop) (resumes the laptop's partial run) and correction rows from r4's own drafts
assert torch.cuda.is_available(), 'Steps 2-3 need a GPU: Runtime > Change runtime type > T4 GPU (step 1 output stays on Drive)'
R4 = 'runs/v10-r4-pre/best_12000.pt'
if not os.path.exists(R4):
    !cat {R4}.part-* > {R4}
for f in ('star_r2.jsonl.gz', 'star_r2.jsonl.gz.done'):
    if not os.path.exists(f'{D}/{f}') and os.path.exists(f'data/{f}'):
        !cp data/{f} {D}/{f}
!python star.py --ckpt {R4} --questions data/symgen/part-k1.jsonl.gz data/symgen/part-k2.jsonl.gz data/symgen/part-k3.jsonl.gz --out {D}/star_r2.jsonl.gz --beam 8 --samples 8 --bs 32
!python build_corrections.py --ckpt {R4} --pool {D}/dataset_v7 --out {D}/corrections.jsonl.gz --n 30000 --extra {D}/dataset_v7/reworded.jsonl.gz {D}/star_r2.jsonl.gz data/teach_rows.jsonl.gz